# 01 — SEC EDGAR Acquisition (Milestone M1)

**DSML stage:** data acquisition & understanding. **Scope:** Nvidia only (the PoC company).

What this notebook does:
1. Builds the **ticker → CIK** mapping for the 14-company universe from SEC `company_tickers.json`
2. Surveys which of the 14 companies actually file with the SEC, and **which forms** (a key data-understanding finding — foreign issuers differ)
3. Downloads Nvidia's recent **10-K** filings (2023+) and the **latest 10-Q** via `edgartools`
4. Persists raw HTML to `data/raw/edgar/NVDA/` with a `manifest.json` for downstream notebooks

**SEC etiquette** (enforced by edgartools): declared identity in the User-Agent, max 10 requests/second.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import edgar

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "Amit Badave amit11badave.ab@gmail.com")
edgar.set_identity(SEC_USER_AGENT)

RAW_EDGAR = PROJECT_ROOT / "data" / "raw" / "edgar"
RAW_EDGAR.mkdir(parents=True, exist_ok=True)
print(f"edgartools {edgar.__version__} | identity set | raw dir: {RAW_EDGAR}")

edgartools 5.40.1 | identity set | raw dir: C:\Users\amit1\OneDrive\Documents\Projects\ai-semiconductor-risk-intelligence-graphrag\data\raw\edgar


C:\Users\amit1\OneDrive\Documents\Projects\ai-semiconductor-risk-intelligence-graphrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Ticker → CIK mapping for the 14-company universe

EDGAR indexes companies by **Central Index Key (CIK)**, not ticker. The official mapping file is
`https://www.sec.gov/files/company_tickers.json`. We persist it raw (immutable input) and derive our universe table.

In [2]:
# The 14-company universe from the feasibility studies, tiered by supply-chain role.
UNIVERSE = {
    # ticker: (company, tier)
    "MSFT": ("Microsoft", "Hyperscaler"),
    "AMZN": ("Amazon", "Hyperscaler"),
    "GOOGL": ("Alphabet", "Hyperscaler"),
    "META": ("Meta", "Hyperscaler"),
    "NVDA": ("Nvidia", "Silicon Designer"),
    "AMD": ("AMD", "Silicon Designer"),
    "AVGO": ("Broadcom", "Silicon Designer"),
    "QCOM": ("Qualcomm", "Silicon Designer"),
    "INTC": ("Intel", "IDM"),
    "TSM": ("TSMC", "Manufacturer"),
    "ASML": ("ASML", "Manufacturer"),
    "MU": ("Micron", "Memory"),
    "SSNLF": ("Samsung", "Memory"),  # OTC ticker; Samsung is NOT an SEC filer — verified below
    "AAPL": ("Apple", "Ecosystem Anchor"),
}

import urllib.request

tickers_path = RAW_EDGAR / "company_tickers.json"
if not tickers_path.exists():
    req = urllib.request.Request(
        "https://www.sec.gov/files/company_tickers.json",
        headers={"User-Agent": SEC_USER_AGENT},
    )
    tickers_path.write_bytes(urllib.request.urlopen(req).read())
raw_tickers = json.loads(tickers_path.read_text())

ticker_to_cik = {row["ticker"]: row["cik_str"] for row in raw_tickers.values()}

universe_df = pd.DataFrame(
    [
        {
            "ticker": t,
            "company": name,
            "tier": tier,
            "cik": ticker_to_cik.get(t),
            "sec_filer": t in ticker_to_cik,
        }
        for t, (name, tier) in UNIVERSE.items()
    ]
)
universe_df

,ticker,company,tier,cik,sec_filer
0,MSFT,Microsoft,Hyperscaler,789019.0,True
1,AMZN,Amazon,Hyperscaler,1018724.0,True
2,GOOGL,Alphabet,Hyperscaler,1652044.0,True
3,META,Meta,Hyperscaler,1326801.0,True
4,NVDA,Nvidia,Silicon Designer,1045810.0,True
5,AMD,AMD,Silicon Designer,2488.0,True
6,AVGO,Broadcom,Silicon Designer,1730168.0,True
7,QCOM,Qualcomm,Silicon Designer,804328.0,True
8,INTC,Intel,IDM,50863.0,True
9,TSM,TSMC,Manufacturer,1046179.0,True


## 2. Filing-form survey — what does each company actually file?

The feasibility docs assume 10-K/10-Q for all 14 companies, but **foreign private issuers file 20-F (annual)
and 6-K (interim) instead**. This matters for the Phase-5 full ingestion design.

In [3]:
survey_rows = []
for _, row in universe_df.iterrows():
    if not row["sec_filer"]:
        survey_rows.append({"ticker": row["ticker"], "company": row["company"], "annual_form": None, "note": "Not an SEC filer"})
        continue
    company = edgar.Company(row["ticker"])
    forms = set(f.form for f in company.get_filings().head(400))
    annual = "10-K" if "10-K" in forms else ("20-F" if "20-F" in forms else "?")
    interim = "10-Q" if "10-Q" in forms else ("6-K" if "6-K" in forms else "?")
    survey_rows.append({
        "ticker": row["ticker"],
        "company": row["company"],
        "annual_form": annual,
        "note": f"interim: {interim}",
    })

survey_df = pd.DataFrame(survey_rows)
survey_df

,ticker,company,annual_form,note
0,MSFT,Microsoft,10-K,interim: 10-Q
1,AMZN,Amazon,10-K,interim: 10-Q
2,GOOGL,Alphabet,10-K,interim: 10-Q
3,META,Meta,10-K,interim: 10-Q
4,NVDA,Nvidia,10-K,interim: 10-Q
5,AMD,AMD,10-K,interim: 10-Q
6,AVGO,Broadcom,10-K,interim: 10-Q
7,QCOM,Qualcomm,10-K,interim: 10-Q
8,INTC,Intel,10-K,interim: 10-Q
9,TSM,TSMC,20-F,interim: 6-K


## 3. Download Nvidia filings (PoC corpus)

Recent 10-Ks (filed 2023 onward — covers the AI capex supercycle per the feasibility docs) plus the latest 10-Q.
Each filing's primary HTML document is persisted under `data/raw/edgar/NVDA/`, and a `manifest.json` records
provenance (accession number, form, dates, source URL, local path) for every downstream notebook.

In [4]:
nvda = edgar.Company("NVDA")
nvda_dir = RAW_EDGAR / "NVDA"
nvda_dir.mkdir(exist_ok=True)

ten_ks = [f for f in nvda.get_filings(form="10-K") if f.filing_date.year >= 2023]
latest_10q = nvda.get_filings(form="10-Q").latest(1)
targets = ten_ks + [latest_10q]

manifest = []
for f in targets:
    local_name = f"{f.form.replace('/', '-')}_{f.filing_date}_{f.accession_no}.html"
    local_path = nvda_dir / local_name
    if not local_path.exists():
        local_path.write_text(f.html(), encoding="utf-8")
    manifest.append({
        "ticker": "NVDA",
        "cik": f.cik,
        "form": f.form,
        "filing_date": str(f.filing_date),
        "accession_no": f.accession_no,
        "source_url": f.document.url,
        "local_path": str(local_path.relative_to(PROJECT_ROOT)),
        "size_bytes": local_path.stat().st_size,
    })

manifest_path = nvda_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
pd.DataFrame(manifest)[["form", "filing_date", "accession_no", "size_bytes"]]

,form,filing_date,accession_no,size_bytes
0,10-K,2026-02-25,0001045810-26-000021,1967824
1,10-K,2025-02-26,0001045810-25-000023,2067528
2,10-K,2024-02-21,0001045810-24-000029,2085571
3,10-K,2023-02-24,0001045810-23-000017,2669243
4,10-Q,2026-05-20,0001045810-26-000052,1167061


## 4. Data dictionary — anatomy of a 10-K

| Item | Title | Why we care (GraphRAG) |
|---|---|---|
| Item 1 | Business | products, segments, customers, suppliers → `Product`, `SUPPLIES_TO` edges |
| **Item 1A** | **Risk Factors** | the core extraction target → `RiskFactor`, `AFFECTED_BY` edges |
| Item 2 | Properties | facilities (post-MVP `Facility` nodes) |
| **Item 7** | **MD&A** | management's narrative on demand, supply constraints, capex |
| Item 8 | Financial Statements | numbers come from **XBRL, never parsed by LLM** (notebook 02) |

**Manifest fields:** `ticker`, `cik`, `form`, `filing_date`, `accession_no` (EDGAR's unique filing id, our graph key),
`source_url` (citation target), `local_path`, `size_bytes`.

**Findings recorded for Phase 5:**
- TSMC and ASML are foreign private issuers → ingest **20-F / 6-K**, not 10-K/10-Q
- **Samsung does not file with the SEC** → it will exist in the graph only as an entity mentioned by others
  (e.g., HBM supplier edges extracted from Nvidia/AMD filings), not as a `Filing` source
- Nvidia's fiscal year ends late January → FY labels ≠ calendar years (handled in notebook 02)

In [5]:
# --- M1 assertion cell: notebook is a self-test under Restart & Run All ---
saved = json.loads(manifest_path.read_text())
assert len(saved) >= 3, f"Expected >=3 filings (2+ 10-Ks + latest 10-Q), got {len(saved)}"
assert any(m["form"] == "10-K" for m in saved) and any(m["form"] == "10-Q" for m in saved)
for m in saved:
    p = PROJECT_ROOT / m["local_path"]
    assert p.exists() and p.stat().st_size > 100_000, f"{p} missing or suspiciously small"
assert (universe_df["sec_filer"].sum() >= 13), "Expected at least 13 of 14 tickers to resolve to a CIK"
print(f"M1 (EDGAR) OK — {len(saved)} Nvidia filings on disk, universe mapped ({universe_df['sec_filer'].sum()}/14 SEC filers)")

M1 (EDGAR) OK — 5 Nvidia filings on disk, universe mapped (13/14 SEC filers)
